<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 3 · Evaluating and Improving Models · Lesson 14 of 24

# Confusion Matrix, Precision and Recall

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Build and read a confusion matrix, calculate precision, recall and F1, and decide which matters more for a real problem.

## Accuracy is not enough

In Lesson 4 a model that said "nobody defaults" scored 79% accuracy and caught **zero** defaulters. To see what a classifier really does, count its four kinds of answers.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 2000
income = rng.lognormal(12.2, 0.45, n).round(-3)
loan = rng.uniform(20_000, 300_000, n).round(-3)
months = rng.integers(1, 61, n)
late = rng.poisson(0.6, n)
salary = rng.random(n) < 0.55
z = -3.2 + 2.5 * (loan / income) - 0.03 * months + 0.8 * late - 0.9 * salary
default = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
loans = pd.DataFrame({"income_rwf": income, "loan_rwf": loan, "months_customer": months,
                      "late_payments": late, "salaried": salary.astype(int),
                      "defaulted": default})
print(loans.shape, "default rate:", round(loans["defaulted"].mean(), 3))
loans.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = loans.drop(columns="defaulted")
y = loans["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

## The confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = model.predict(X_test)
print(confusion_matrix(y_test, pred))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["repaid", "default"])
plt.show()

You should see:

|  | Predicted repaid | Predicted default |
|---|---|---|
| **Really repaid** | 366 (TN) | 28 (FP) |
| **Really defaulted** | 54 (FN) | 52 (TP) |

- **True positives (TP) = 52** — defaulters correctly flagged.
- **True negatives (TN) = 366** — good borrowers correctly approved.
- **False positives (FP) = 28** — good borrowers wrongly flagged (a lost customer).
- **False negatives (FN) = 54** — defaulters the model missed (lost money).

"Positive" simply means the class we are looking for (default), not something good.

## Precision and recall

> **Precision** = TP ÷ (TP + FP) — of the borrowers we flagged, how many really defaulted?
> **Recall** = TP ÷ (TP + FN) — of all real defaulters, how many did we catch?

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision:", round(precision_score(y_test, pred), 3))   # 52 / 80
print("Recall:   ", round(recall_score(y_test, pred), 3))      # 52 / 106
print("F1:       ", round(f1_score(y_test, pred), 3))

- Precision is about **0.65**: when the model says "default", it is right 65% of the time.
- Recall is about **0.49**: it catches only **half** of the real defaulters.

The 84% accuracy hid this weakness completely.

## F1: one number for both

> F1 = 2 × precision × recall ÷ (precision + recall)

F1 (here about **0.56**) is high only when **both** precision and recall are high. It is useful when the positive class is rare.

## The full report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred, target_names=["repaid", "default"], digits=3))

It shows precision, recall and F1 for each class, plus their averages. **Support** is the number of real examples of each class.

## Which matters more?

| Situation | Worse mistake | Focus on |
|---|---|---|
| Cancer screening | Missing a cancer (FN) | **Recall** |
| Fraud alerts on mobile money | Missing fraud (FN), but too many alerts annoy customers | Recall, while watching precision |
| Spam filter | Hiding a real email (FP) | **Precision** |
| Approving micro-loans | Depends on the money lost vs earned | Both — choose with costs (next lesson) |

There is almost always a **trade-off**: catching more positives (higher recall) usually means more false alarms (lower precision). The decision threshold controls it — next lesson.

## Sensitivity and specificity

Doctors use other names:

- **Sensitivity** = recall = TP ÷ (TP + FN).
- **Specificity** = TN ÷ (TN + FP) — how many healthy people are correctly cleared. Here 366 ÷ 394 ≈ **0.93**.

## Key terms

- **Confusion matrix** — table of TP, TN, FP and FN.
- **Precision** — share of predicted positives that are correct.
- **Recall (sensitivity)** — share of real positives that are found.
- **F1 score** — balance of precision and recall.
- **Specificity** — share of real negatives correctly identified.

**Takeaway:** Always look at the confusion matrix; precision and recall show which mistakes a model makes, and the right balance depends on what each mistake costs.

## Try it yourself

1. Compute precision and recall by hand from the confusion matrix, then check with scikit-learn.
2. Build the confusion matrix for the "nobody defaults" `DummyClassifier`. What are its precision and recall?
3. Train a random forest and compare its precision and recall with logistic regression.
4. For a mobile-money fraud detector, write two sentences on whether you would favour precision or recall, and why.

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*